# dbstim walkthrough

Two journeys, one after the other:

- **A. Download videos for your own code:** create an experiment from filters, a count and a seed, then download a portable BIDS folder.
- **B. Track a run:** check a subject session out to this computer, record a run offline, then sync it.

Run the cells one at a time. Set the `DBSTIM_URL` environment variable to try a local test server instead of dbstimuli.com.

## 1. Connect
`connect()` asks for your dbstimuli.com username and password; nothing is saved. `metrics()` lists the metric names you can filter on.

In [ ]:
import os
import time

import dbstim
from dbstim import Filter

client = dbstim.connect(url=os.environ.get("DBSTIM_URL", "https://dbstimuli.com"))
client.metrics()

## A. Download videos for your own code
`create()` saves a frozen selection: every video matching all filters, sampled by `count` and `seed`. Names are unique in your account; to rerun this cell with the same name, use `client.open(NAME)` instead.

In [ ]:
NAME = "Notebook demo"
experiment = client.create(NAME, filters=[Filter("duration_seconds", ">=", 1)], count=4, seed=1)
experiment.name, experiment.revision, experiment.videos

`download()` writes the videos under `stimuli/` with `manifest.json` (revision and checksums) and `manifest.csv` (for PGL). Running it again reuses verified files. No subjects, scheduling or tracking are involved.

In [ ]:
folder = experiment.download("notebook_demo_bids")
sorted(path.relative_to(folder).as_posix() for path in folder.rglob("*") if path.is_file())

In [ ]:
print((folder / "stimuli" / "manifest.csv").read_text())

## B. Track a run
**Online, before the session.** `schedule()` saves a new revision with subjects, sessions and runs. `checkout()` reserves one subject session for this computer and downloads only its videos. Another computer would need `takeover=True`.

In [ ]:
scheduled = experiment.schedule(subjects=1, runs=2, trials_per_run=2)
checked = scheduled.checkout(subject=1, session=1, folder="notebook_sub01_ses01")
checked.subject, checked.session, checked.runs

**Offline, during the run.** `dbstim.session(folder)` reopens the checkout without the network. Inside `with session.run(n)`, each call is written to disk before it returns, and iterating gives the trials that have no outcome yet.

A real task plays `trial.file` with PGL (see `examples/pgl_tracked_movie_task.py`). Here, stand-in frame times feed `frame_summary()`, whose output goes into the result.

In [ ]:
session = dbstim.session("notebook_sub01_ses01")
with session.run(1) as run:
    t0 = time.monotonic()
    for trial in run:
        run.started(trial, onset=time.monotonic() - t0)
        frames = [i / 30 for i in range(30)]  # stand-in for PGL's movieTimes
        evidence = dbstim.frame_summary([t0 + t for t in frames], frames, 1.0)
        run.completed(trial, duration=evidence["seconds_shown"], result=evidence)

Leaving the block wrote a BIDS events file and sidecar from the journal. If the program had crashed, reopening the run would record the open trial as abandoned and continue with the next one.

In [ ]:
events = next(session.folder.glob("sub-*/ses-*/beh/*_run-01_events.tsv"))
print(events.read_text())

**Online, afterwards.** `sync()` uploads each finished run once: `accepted`, `already synced`, `kept as flagged attempt`, or `waiting` for a run not finished yet. Run 2 has not been recorded, so it waits.

In [ ]:
session.sync(client=client)

In [ ]:
client.close()